In [1]:
import yaml
import os
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
import pydantic
pydantic.__version__

'2.12.5'

In [3]:
from app.model.info import InfoPartial

In [4]:
import os
from app.yaml_client import get_yaml_str, yaml_to_dict
from app.core.config import get_config

yaml_str = get_yaml_str()
yaml_dict = yaml_to_dict(yaml_str)
yaml_dict

{'openapi': '3.0.2',
 'info': {'description': 'Create beautiful product and API documentation with our developer friendly platform.',
  'version': '4.355.0',
  'title': 'ReadMe API 🦉',
  'contact': {'name': 'API Support',
   'url': 'https://docs.readme.com/main/docs/need-more-support',
   'email': 'support@readme.io'},
  'license': {'name': 'Apache 2.0',
   'url': 'https://www.apache.org/licenses/LICENSE-2.0.html'}},
 'servers': [{'url': 'https://dash.readme.com/api/v1'}],
 'tags': [{'name': 'API Registry'},
  {'name': 'API Specification'},
  {'name': 'Apply to ReadMe'},
  {'name': 'Categories'},
  {'name': 'Changelog'},
  {'name': 'Custom Pages'},
  {'name': 'Docs'},
  {'name': 'Errors'},
  {'name': 'Projects'},
  {'name': 'Version'}],
 'paths': {'/api-registry/{uuid}': {'get': {'operationId': 'getAPIRegistry',
    'summary': 'Retrieve an entry from the API Registry',
    'description': "Get an API definition file that's been uploaded to ReadMe.",
    'tags': ['API Registry'],
    'pa

In [5]:
from datetime import date
import datetime

PRIMITIVES = (str, int, float, bool, type(None), date, datetime, datetime.time)

info_primitives = yaml_dict["info"].items()
info_primitives
{key:value for key, value in info_primitives if type(value) in PRIMITIVES}

{'description': 'Create beautiful product and API documentation with our developer friendly platform.',
 'version': '4.355.0',
 'title': 'ReadMe API 🦉'}

In [6]:
info = {
        "description": "Create beautiful product and API documentation with our developer friendly platform.",
        "version": "4.355.0",
        "title": "ReadMe API 🦉",
        "contact": {
            "name": "API Support",
            "url": "https://docs.readme.com/main/docs/need-more-support",
            "email": "support@readme.io"
        }
    }
info = InfoPartial(**info)
dict(info)

{'title': 'ReadMe API 🦉',
 'description': 'Create beautiful product and API documentation with our developer friendly platform.',
 'termsOfService': None,
 'contact': Contact(name='API Support', url='https://docs.readme.com/main/docs/need-more-support', email='support@readme.io'),
 'license': None,
 'version': '4.355.0'}

In [8]:
from app.model.server import ServerPartial
from typing import List
from datetime import date
import datetime
from neo4j import GraphDatabase, Driver, Session
from app.model.spec import SpecPartial

URI = "bolt://localhost:7687"
USER = "neo4j"
PASSWORD = "password"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

PRIMITIVES = (str, int, float, bool, type(None), date, datetime, datetime.time)


def create_info_nodes(session: Session, title: str, info: InfoPartial):
  info_primitives = {key: value for key, value in dict(info).items() if type(value) in PRIMITIVES}

  session.run("MATCH (a:Api { title: $title }) MERGE (a)-[:HAS_INFO]->(i:Info { title: $title });", title=title)
  session.run("MERGE (i: Info { title: $title }) SET i += $info;", title=title, info=info_primitives)

  if info.contact:
    session.run("MATCH (a:Api { title: $title }) MERGE (a)-[:HAS_CONTACT]->(co: Contact { title: $title })", title=title)
    session.run("MERGE (co: Contact { title: $title }) SET co += $contact;", title=title, contact=dict(info.contact))

  if info.license:
    session.run("MATCH (a:Api { title: $title }) MERGE (a)-[:HAS_LICENSE]->(l: License { title: $title })", title=title)
    session.run("MERGE (l: License { title: $title }) SET l += $license;", title=title, license=dict(info.license))

def create_server_nodes(session: Session, title: str, servers: ServerPartial | None = None):
  if servers is None:
      servers = []
  for server in servers:
      create_server_node(session, title, server)

def create_server_node(session:Session, title: str, server: ServerPartial):
  session.run("MATCH (a:Api { title: $title }) MERGE (a)-[:HAS_SERVER]->(s: Server { url: $server.url, title: $title })", title=title, server=server)
  session.run("""
    MERGE (s: Server { url: $server.url, title: $title })
      SET s.description = $server.description,
        s += $server.variables
    """, server=server, title=title)

def create_graph(driver: Driver, open_api_spec: dict):
  spec = SpecPartial(**open_api_spec)
  errors: List[Exception] = []
  info = InfoPartial(**open_api_spec.get('info') or {})
  title = info.title if info.title else 'oas-spec'

  output = (title, errors)

  with driver.session() as session:
    # Central node
    session.run("MERGE (a:Api { title: $title })", title=title)

    # Openapi spec version
    if open_api_spec.get('openapi'):
      session.run("MERGE (a:Api { title: $title }) SET a.openapi = $openapi", title=title,
                  openapi=open_api_spec.get('openapi'))

    # Info
    create_info_nodes(session, title, info)

    # Servers
    create_server_nodes(session, title, spec.servers)

    session.close()

    return output


create_graph(driver, yaml_dict)


ValueError: Values of type <class 'app.model.server.Server'> are not supported

In [10]:
import torch

tensor = torch.Tensor([[61,100],[36+14,77+14]])
tensor

tensor([[ 61., 100.],
        [ 50.,  91.]])

In [11]:
tensor * 2.54

tensor([[154.9400, 254.0000],
        [127.0000, 231.1400]])